<a href="https://colab.research.google.com/github/kartikacandrak/dysgraphiaBaseModel/blob/main/dysgraphia_modality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Online Handwriting Feature**

In [ ]:
import os
import numpy as np
import pandas as pd


# =========================
# SAFE STATISTICS
# =========================
def safe_stats(x):
    x = np.nan_to_num(np.array(x), nan=0.0, posinf=0.0, neginf=0.0)

    if len(x) < 2:
        return [0]*7

    return [
        float(np.mean(x)),
        float(np.median(x)),
        float(np.std(x)),
        float(np.min(x)),
        float(np.max(x)),
        float(np.percentile(x, 5)),
        float(np.percentile(x, 95))
    ]


# =========================
# DERIVATIVE HELPER
# =========================
def diff(arr, t):
    arr = np.array(arr)
    t = np.array(t)

    if len(arr) < 2:
        return np.zeros(max(len(arr)-1, 1))

    dt = np.diff(t) + 1e-6
    return np.diff(arr) / dt


# =========================
# LOAD FILE
# =========================
def load_txt(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:, 0].values
    y = df.iloc[:, 1].values
    t = df.iloc[:, 2].values + 1e-6

    status = df.iloc[:, 3].values

    az = df.iloc[:, 4].values
    alt = df.iloc[:, 5].values
    pres = df.iloc[:, 6].values

    return x, y, t, status, az, alt, pres


# =========================
# FEATURE ENGINE (CORE)
# =========================
def build_features(x, y, t, az, alt, pres, prefix):

    if len(x) < 4:
        return {}

    # =========================
    # POSITION DERIVATIVES
    # =========================
    vx = diff(x, t)
    vy = diff(y, t)

    ax = diff(vx, t[:-1])
    ay = diff(vy, t[:-1])

    jx = diff(ax, t[:-2])
    jy = diff(ay, t[:-2])

    # =========================
    # MAGNITUDE (IMPORTANT ADDITION)
    # =========================
    r  = np.sqrt(x**2 + y**2)
    vr = np.sqrt(vx**2 + vy**2)
    ar = np.sqrt(ax**2 + ay**2)

    # =========================
    # SENSOR DERIVATIVES
    # =========================
    az1 = diff(az, t)
    az2 = diff(az1, t[:-1])

    alt1 = diff(alt, t)
    alt2 = diff(alt1, t[:-1])

    pres1 = diff(pres, t)
    pres2 = diff(pres1, t[:-1])

    feats = {}

    def add(name, arr):
        s = safe_stats(arr)
        feats.update({
            f"{prefix}_{name}_mean": s[0],
            f"{prefix}_{name}_median": s[1],
            f"{prefix}_{name}_std": s[2],
            f"{prefix}_{name}_min": s[3],
            f"{prefix}_{name}_max": s[4],
            f"{prefix}_{name}_p5": s[5],
            f"{prefix}_{name}_p95": s[6],
        })

    # =========================
    # POSITION
    # =========================
    add("x", x)
    add("y", y)

    # =========================
    # VELOCITY
    # =========================
    add("vx", vx)
    add("vy", vy)
    add("vr", vr)

    # =========================
    # ACCELERATION
    # =========================
    add("ax", ax)
    add("ay", ay)
    add("ar", ar)

    # =========================
    # JERK
    # =========================
    add("jx", jx)
    add("jy", jy)

    # =========================
    # SENSOR RAW
    # =========================
    add("az", az)
    add("alt", alt)
    add("pres", pres)

    # =========================
    # SENSOR 1st DERIVATIVE
    # =========================
    add("az1", az1)
    add("alt1", alt1)
    add("pres1", pres1)

    # =========================
    # SENSOR 2nd DERIVATIVE
    # =========================
    add("az2", az2)
    add("alt2", alt2)
    add("pres2", pres2)

    return feats


# =========================
# MAIN FEATURE EXTRACTION
# =========================
def extract_features(path):

    x, y, t, status, az, alt, pres = load_txt(path)

    s1 = status == 1
    s0 = status == 0

    # =========================
    # SURFACE & AIR SPLIT
    # =========================
    surface = build_features(
        x[s1], y[s1], t[s1],
        az[s1], alt[s1], pres[s1],
        "surface"
    )

    air = build_features(
        x[s0], y[s0], t[s0],
        az[s0], alt[s0], pres[s0],
        "air"
    )

    # =========================
    # GLOBAL FEATURES
    # =========================
    r = np.sqrt(x**2 + y**2)
    vr = np.sqrt(diff(x,t)**2 + diff(y,t)**2)

    global_feat = {
        "width": np.max(x) - np.min(x),
        "height": np.max(y) - np.min(y),
        "duration": t[-1] - t[0],
        "length": np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2)),

        "pen_lift": np.sum((status[:-1]==1) & (status[1:]==0)),

        "r_mean": np.mean(r),
        "vr_mean": np.mean(vr)
    }

    return {**surface, **air, **global_feat}


# =========================
# DATASET PIPELINE
# =========================
def process_dataset(root_folder, output_csv):

    rows = []

    for root, _, files in os.walk(root_folder):

        parts = root.split(os.sep)

        task = parts[-2] if len(parts) >= 2 else "task"
        grade = parts[-1] if len(parts) >= 1 else "grade"

        for file in files:

            if not file.endswith(".txt"):
                continue

            path = os.path.join(root, file)

            try:
                feat = extract_features(path)

                feat["filename"] = file
                feat["task"] = task
                feat["grade"] = grade

                rows.append(feat)

            except:
                continue

    df = pd.DataFrame(rows)

    # =========================
    # CLEAN DATA
    # =========================
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    df = df[["filename", "task", "grade"] +
            [c for c in df.columns if c not in ["filename","task","grade"]]]

    df.to_csv(output_csv, index=False)

    print("FINAL SHAPE:", df.shape)
    print("FEATURE COUNT:", df.shape[1]-3)

    return df


# =========================
# RUN
# =========================
df = process_dataset(
    "/content/drive/MyDrive/DysgraphiaDB3/Online",
    "/content/drive/MyDrive/DysgraphiaDB3/features_231_FINAL.csv"
)

OK: user00017_3.txt TASK: grade1
OK: user00017_2.txt TASK: grade1
OK: user00036_4.txt TASK: grade1
OK: user00016_3.txt TASK: grade1
OK: user00007_3.txt TASK: grade1
OK: user00014_4.txt TASK: grade1
OK: user00036_1.txt TASK: grade1
OK: user00044_1.txt TASK: grade1
OK: user00007_2.txt TASK: grade1
OK: user00014_2.txt TASK: grade1
OK: user00042_2.txt TASK: grade1
OK: user00015_3.txt TASK: grade1
OK: user00007_1.txt TASK: grade1
OK: user00015_1.txt TASK: grade1
OK: user00014_3.txt TASK: grade1
OK: user00044_3.txt TASK: grade1
OK: user00036_3.txt TASK: grade1
OK: user00042_3.txt TASK: grade1
OK: user00036_6.txt TASK: grade1
OK: user00019_3.txt TASK: grade1
OK: user00015_2.txt TASK: grade1
OK: user00019_1.txt TASK: grade1
OK: user00017_1.txt TASK: grade1
OK: user00019_2.txt TASK: grade1
OK: user00044_2.txt TASK: grade1
OK: user00014_1.txt TASK: grade1
OK: user00011_4.txt TASK: grade1
OK: user00042_1.txt TASK: grade1
OK: user00036_7.txt TASK: grade1
OK: user00029_2.txt TASK: grade1
OK: user00

**Modalitas Tunggal (Online Handwriting)**

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat


def load_data():

    Xo, paths, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue


                try:
                    feat = extract_online_features(os.path.join(o_path,f))
                    Xo.append(feat)
                    paths.append(os.path.join(o_path, f))
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), np.array(paths), np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online,paths, y= load_data()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(paths):
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        svm = SVC(**svm_online_params[task], probability=True)
        svm.fit(Xtr, y_t[tr])
        pred = svm.predict(Xte)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = svm.predict_proba(Xte)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Modalitas/online_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Modalitas/online_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Modalitas/online_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")


TASK: word
word total data: 395
Fold 1 | Acc:0.850 | Kappa:0.749 | ROC:0.977
Fold 2 | Acc:0.700 | Kappa:0.521 | ROC:0.903
Fold 3 | Acc:0.850 | Kappa:0.756 | ROC:0.964
Fold 4 | Acc:0.825 | Kappa:0.717 | ROC:0.933
Fold 5 | Acc:0.800 | Kappa:0.681 | ROC:0.952
Fold 6 | Acc:0.718 | Kappa:0.517 | ROC:0.929
Fold 7 | Acc:0.744 | Kappa:0.571 | ROC:0.930
Fold 8 | Acc:0.692 | Kappa:0.479 | ROC:0.887
Fold 9 | Acc:0.821 | Kappa:0.704 | ROC:0.937
Fold 10 | Acc:0.821 | Kappa:0.701 | ROC:0.955

FINAL RESULT: word
Accuracy : 0.7822784810126582
Kappa    : 0.6406507849193923

TASK: pword
pword total data: 350
Fold 1 | Acc:0.800 | Kappa:0.645 | ROC:0.909
Fold 2 | Acc:0.857 | Kappa:0.756 | ROC:0.964
Fold 3 | Acc:0.886 | Kappa:0.806 | ROC:0.960
Fold 4 | Acc:0.686 | Kappa:0.449 | ROC:0.868
Fold 5 | Acc:0.686 | Kappa:0.472 | ROC:0.898
Fold 6 | Acc:0.686 | Kappa:0.473 | ROC:0.862
Fold 7 | Acc:0.771 | Kappa:0.616 | ROC:0.888
Fold 8 | Acc:0.771 | Kappa:0.617 | ROC:0.935
Fold 9 | Acc:0.657 | Kappa:0.407 | ROC:0.

**Modalitas Tunggal : Offline Handwriting**

In [ ]:
# =========================================================
# OFFLINE DYSGRAPHIA - CNN + SVM
# STRATIFIED 10-FOLD CV + COMPLETE LOGGING
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER
# =========================
svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# CNN FEATURE EXTRACTOR
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# =========================
# CNN EXTRACTION SAFE
# =========================
def extract_cnn(paths, model):
    model.eval()
    feats = []

    with torch.no_grad():
        for p in paths:
            try:
                img = Image.open(p).convert("RGB")
                img = transform(img).unsqueeze(0).to(device)
                feats.append(model(img).cpu().numpy().flatten())
            except:
                feats.append(np.zeros(1920))  # FIX SAFE PAD

    return np.array(feats)

# =========================
# LOAD DATA
# =========================
def load_data():
    Xi, y = [], []

    label_map = {"normal":0,"grade1":1,"grade2":2}

    for task in ["word","pword","dword"]:
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(offline_path):
            cls_path = os.path.join(offline_path, cls)

            for f in os.listdir(cls_path):
                if not f.endswith(".png"):
                    continue

                Xi.append(os.path.join(cls_path, f))
                y.append(label_map[cls])

    return Xi, np.array(y)

# =========================
# INIT
# =========================
X_img, y = load_data()
cnn = CNNFeatureExtractor().to(device)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:", task)
    print("====================")

    idx = [i for i,p in enumerate(X_img) if task in p]

    if len(idx) == 0:
        continue

    imgs = [X_img[i] for i in idx]
    y_t = y[idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(imgs,y_t),1):

        # =========================
        # CNN FEATURE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr], cnn)
        Xe = extract_cnn([imgs[i] for i in te], cnn)

        y_tr = y_t[tr]
        y_te = y_t[te]

        # =========================
        # SCALE
        # =========================
        sc = StandardScaler()
        Xt = sc.fit_transform(Xt)
        Xe = sc.transform(Xe)

        # =========================
        # SVM
        # =========================
        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt, y_tr)

        pred = svm_c.predict(Xe)
        proba = svm_c.predict_proba(Xe)

        y_true = y_te

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true, pred)
        prec = precision_score(y_true, pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, pred, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_true, pred)

        cm = confusion_matrix(y_true, pred)

        # =========================
        # ROC SAFE
        # =========================
        try:
            if len(np.unique(y_true)) > 1:
                roc = roc_auc_score(y_true, proba, multi_class='ovr', average='weighted')
            else:
                roc = np.nan
        except:
            roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | F1:{f1:.3f} | ROC:{roc}")

        # =========================
        # SAVE RESULTS
        # =========================
        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # CM PER FOLD SAVE
        pd.DataFrame(cm).to_csv(
            f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Modalitas/Offline/{task}_fold{fold}_cm.csv",
            index=False
        )

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL TASK RESULT
    # =========================
    print("\nFINAL RESULT:", task)
    print("Accuracy:", accuracy_score(y_true_all, y_pred_all))
    print("Kappa   :", cohen_kappa_score(y_true_all, y_pred_all))

    # =========================
    # CM TOTAL PER TASK
    # =========================
    cms_task = [c["cm"] for c in cms if c["task"] == task]

    if len(cms_task) > 0:
        cm_total = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_total).to_csv(
            f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Modalitas/Offline/{task}_cm_total.csv",
            index=False
        )

# =========================
# SAVE ALL METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Modalitas/Offline/metrics_per_fold.csv", index=False)

# =========================
# FINAL SUMMARY
# =========================
print("\n===== FINAL MEAN RESULT =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")


TASK: word
Fold 1 | Acc:0.546 | F1:0.407 | ROC:0.7085875090777052
Fold 2 | Acc:0.565 | F1:0.444 | ROC:0.8550556765916243
Fold 3 | Acc:0.574 | F1:0.461 | ROC:0.8190777051561364
Fold 4 | Acc:0.602 | F1:0.508 | ROC:0.8960869038973613
Fold 5 | Acc:0.565 | F1:0.441 | ROC:0.8333914306463326
Fold 6 | Acc:0.602 | F1:0.503 | ROC:0.8844661035304311
Fold 7 | Acc:0.602 | F1:0.515 | ROC:0.7694439029672948
Fold 8 | Acc:0.602 | F1:0.506 | ROC:0.7638331167424736
Fold 9 | Acc:0.579 | F1:0.468 | ROC:0.7889971647590045
Fold 10 | Acc:0.551 | F1:0.412 | ROC:0.8096027162308796

FINAL RESULT: word
Accuracy: 0.5788497217068646
Kappa   : 0.1306976628525348

TASK: pword
Fold 1 | Acc:0.714 | F1:0.697 | ROC:0.8827948420747609
Fold 2 | Acc:0.829 | F1:0.820 | ROC:0.9359074664348498
Fold 3 | Acc:0.743 | F1:0.716 | ROC:0.932374674007534
Fold 4 | Acc:0.800 | F1:0.778 | ROC:0.9021491355162754
Fold 5 | Acc:0.771 | F1:0.759 | ROC:0.9590738916256157
Fold 6 | Acc:0.714 | F1:0.669 | ROC:0.9732315270935961
Fold 7 | Acc:0.88

**Multimodal+ Dence121+ Meta Learning**

| No | Kelompok Fitur              | Sumber Data                                  | Deskripsi                                                  | Statistik yang Digunakan                                                                   | Jumlah Fitur |
| -- | --------------------------- | -------------------------------------------- | ---------------------------------------------------------- | ------------------------------------------------------------------------------------------ | ------------ |
| 1  | Surface Features (Pen Down) | x, y, t (pen_status = 1)                     | Velocity, acceleration, jerk, speed magnitude saat menulis | Mean, Median, Std, Min, Max, 5th Percentile, 95th Percentile                               | 77           |
| 2  | Air Features (Pen Up)       | x, y, t (pen_status = 0)                     | Gerakan transisi saat pena tidak menyentuh permukaan       | Mean, Median, Std, Min, Max, 5th Percentile, 95th Percentile                               | 77           |
| 3  | Sensor Features             | altitude, azimuth, pressure (pen_status = 1) | Statistik orientasi dan tekanan stylus                     | Mean, Median, Std, Min, Max, 5th Percentile, 95th Percentile                               | 21           |
| 4  | Global Features             | x, y, t                                      | Karakteristik keseluruhan tulisan                          | (Tidak memakai statistik, nilai langsung) Width, Height, Duration, Total Length, Pen Lifts | 5            |
|    |                             |                                              |                                                            | **TOTAL FITUR**                                                                            | **194**      |


Dense201

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat

# =========================
# CNN FEATURE
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.densenet201(pretrained=True)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(paths, model):
    model.eval()
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())
    return np.array(feats)

def load_data():

    Xo, Xi, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)
            i_path = os.path.join(offline_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue

                base = os.path.splitext(f)[0]
                img = os.path.join(i_path, base + ".png")

                if not os.path.exists(img):
                    continue

                try:
                    feat = extract_online_features(os.path.join(o_path,f))

                    Xo.append(feat)
                    Xi.append(img)
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), Xi, np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online, X_img, y = load_data()

cnn = CNNFeatureExtractor().to(device)

#skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(X_img):
      #parts = p.split("/")
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]
    imgs = [X_img[i] for i in idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])

        svm_o = SVC(**svm_online_params[task], probability=True)
        svm_o.fit(Xtr,y_t[tr])

        po_tr = svm_o.predict_proba(Xtr)
        po_te = svm_o.predict_proba(Xte)

        # =========================
        # OFFLINE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr],cnn)
        Xe = extract_cnn([imgs[i] for i in te],cnn)

        sc2 = StandardScaler()
        Xt = sc2.fit_transform(Xt)
        Xe = sc2.transform(Xe)

        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt,y_t[tr])

        pc_tr = svm_c.predict_proba(Xt)
        pc_te = svm_c.predict_proba(Xe)

        # =========================
        # META
        # =========================
        meta_tr = np.hstack([po_tr,pc_tr])
        meta_te = np.hstack([po_te,pc_te])

        meta = SVC(kernel='rbf',C=1,probability=True)
        meta.fit(meta_tr,y_t[tr])

        pred = meta.predict(meta_te)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = meta.predict_proba(meta_te)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Dense201/Dense201_cm_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Dense201/Dense201_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Dense201/Dense201_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")

Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth


100%|██████████| 77.4M/77.4M [00:00<00:00, 118MB/s]



TASK: word
word total data: 394
Fold 1 | Acc:0.825 | Kappa:0.705 | ROC:0.952
Fold 2 | Acc:0.750 | Kappa:0.598 | ROC:0.842
Fold 3 | Acc:0.800 | Kappa:0.666 | ROC:0.946
Fold 4 | Acc:0.825 | Kappa:0.719 | ROC:0.947
Fold 5 | Acc:0.821 | Kappa:0.723 | ROC:0.940
Fold 6 | Acc:0.846 | Kappa:0.741 | ROC:0.972
Fold 7 | Acc:0.821 | Kappa:0.707 | ROC:0.917
Fold 8 | Acc:0.769 | Kappa:0.627 | ROC:0.908
Fold 9 | Acc:0.821 | Kappa:0.706 | ROC:0.943
Fold 10 | Acc:0.769 | Kappa:0.610 | ROC:0.939

FINAL RESULT: word
Accuracy : 0.8045685279187818
Kappa    : 0.6800632744529396

TASK: pword
pword total data: 350
Fold 1 | Acc:0.829 | Kappa:0.695 | ROC:0.956
Fold 2 | Acc:0.914 | Kappa:0.859 | ROC:0.994
Fold 3 | Acc:0.857 | Kappa:0.755 | ROC:0.990
Fold 4 | Acc:0.829 | Kappa:0.714 | ROC:0.967
Fold 5 | Acc:0.800 | Kappa:0.657 | ROC:0.940
Fold 6 | Acc:0.829 | Kappa:0.708 | ROC:0.950
Fold 7 | Acc:0.771 | Kappa:0.621 | ROC:0.942
Fold 8 | Acc:0.857 | Kappa:0.758 | ROC:0.974
Fold 9 | Acc:0.943 | Kappa:0.905 | ROC:0.

**DenseNet121**

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat

# =========================
# CNN FEATURE
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.densenet121(pretrained=True)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(paths, model):
    model.eval()
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())
    return np.array(feats)

def load_data():

    Xo, Xi, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)
            i_path = os.path.join(offline_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue

                base = os.path.splitext(f)[0]
                img = os.path.join(i_path, base + ".png")

                if not os.path.exists(img):
                    continue

                try:
                    feat = extract_online_features(os.path.join(o_path,f))

                    Xo.append(feat)
                    Xi.append(img)
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), Xi, np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online, X_img, y = load_data()

cnn = CNNFeatureExtractor().to(device)

#skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(X_img):
      #parts = p.split("/")
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]
    imgs = [X_img[i] for i in idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])

        svm_o = SVC(**svm_online_params[task], probability=True)
        svm_o.fit(Xtr,y_t[tr])

        po_tr = svm_o.predict_proba(Xtr)
        po_te = svm_o.predict_proba(Xte)

        # =========================
        # OFFLINE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr],cnn)
        Xe = extract_cnn([imgs[i] for i in te],cnn)

        sc2 = StandardScaler()
        Xt = sc2.fit_transform(Xt)
        Xe = sc2.transform(Xe)

        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt,y_t[tr])

        pc_tr = svm_c.predict_proba(Xt)
        pc_te = svm_c.predict_proba(Xe)

        # =========================
        # META
        # =========================
        meta_tr = np.hstack([po_tr,pc_tr])
        meta_te = np.hstack([po_te,pc_te])

        meta = SVC(kernel='rbf',C=1,probability=True)
        meta.fit(meta_tr,y_t[tr])

        pred = meta.predict(meta_te)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = meta.predict_proba(meta_te)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Dense121/Dense121_cm_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Dense121/Dense121_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Dense121/Dense121_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 69.4MB/s]



TASK: word
word total data: 394
Fold 1 | Acc:0.825 | Kappa:0.705 | ROC:0.962
Fold 2 | Acc:0.750 | Kappa:0.598 | ROC:0.845
Fold 3 | Acc:0.800 | Kappa:0.666 | ROC:0.948
Fold 4 | Acc:0.800 | Kappa:0.677 | ROC:0.950
Fold 5 | Acc:0.821 | Kappa:0.723 | ROC:0.930
Fold 6 | Acc:0.846 | Kappa:0.741 | ROC:0.968
Fold 7 | Acc:0.821 | Kappa:0.707 | ROC:0.914
Fold 8 | Acc:0.769 | Kappa:0.627 | ROC:0.919
Fold 9 | Acc:0.795 | Kappa:0.661 | ROC:0.942
Fold 10 | Acc:0.769 | Kappa:0.610 | ROC:0.939

FINAL RESULT: word
Accuracy : 0.799492385786802
Kappa    : 0.6713026030941444

TASK: pword
pword total data: 350
Fold 1 | Acc:0.829 | Kappa:0.701 | ROC:0.942
Fold 2 | Acc:0.943 | Kappa:0.906 | ROC:0.999
Fold 3 | Acc:0.914 | Kappa:0.854 | ROC:0.992
Fold 4 | Acc:0.829 | Kappa:0.703 | ROC:0.976
Fold 5 | Acc:0.857 | Kappa:0.761 | ROC:0.959
Fold 6 | Acc:0.857 | Kappa:0.765 | ROC:0.939
Fold 7 | Acc:0.829 | Kappa:0.720 | ROC:0.991
Fold 8 | Acc:0.914 | Kappa:0.854 | ROC:0.998
Fold 9 | Acc:0.914 | Kappa:0.855 | ROC:0.9

**Efficient Net B7**

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat

# =========================
# CNN FEATURE
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.efficientnet_b7(pretrained=True)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(paths, model):
    model.eval()
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())
    return np.array(feats)

def load_data():

    Xo, Xi, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)
            i_path = os.path.join(offline_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue

                base = os.path.splitext(f)[0]
                img = os.path.join(i_path, base + ".png")

                if not os.path.exists(img):
                    continue

                try:
                    feat = extract_online_features(os.path.join(o_path,f))

                    Xo.append(feat)
                    Xi.append(img)
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), Xi, np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online, X_img, y = load_data()

cnn = CNNFeatureExtractor().to(device)

#skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(X_img):
      #parts = p.split("/")
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]
    imgs = [X_img[i] for i in idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])

        svm_o = SVC(**svm_online_params[task], probability=True)
        svm_o.fit(Xtr,y_t[tr])

        po_tr = svm_o.predict_proba(Xtr)
        po_te = svm_o.predict_proba(Xte)

        # =========================
        # OFFLINE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr],cnn)
        Xe = extract_cnn([imgs[i] for i in te],cnn)

        sc2 = StandardScaler()
        Xt = sc2.fit_transform(Xt)
        Xe = sc2.transform(Xe)

        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt,y_t[tr])

        pc_tr = svm_c.predict_proba(Xt)
        pc_te = svm_c.predict_proba(Xe)

        # =========================
        # META
        # =========================
        meta_tr = np.hstack([po_tr,pc_tr])
        meta_te = np.hstack([po_te,pc_te])

        meta = SVC(kernel='rbf',C=1,probability=True)
        meta.fit(meta_tr,y_t[tr])

        pred = meta.predict(meta_te)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = meta.predict_proba(meta_te)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/EfficientB7/EfficientB7_cm_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/EfficientB7/EfficientB7_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/EfficientB7/EfficienNetB7_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B7_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B7_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



TASK: word
word total data: 394
Fold 1 | Acc:0.875 | Kappa:0.792 | ROC:0.958
Fold 2 | Acc:0.700 | Kappa:0.511 | ROC:0.859
Fold 3 | Acc:0.825 | Kappa:0.710 | ROC:0.949
Fold 4 | Acc:0.850 | Kappa:0.765 | ROC:0.953
Fold 5 | Acc:0.821 | Kappa:0.717 | ROC:0.920
Fold 6 | Acc:0.846 | Kappa:0.743 | ROC:0.973
Fold 7 | Acc:0.795 | Kappa:0.668 | ROC:0.915
Fold 8 | Acc:0.795 | Kappa:0.668 | ROC:0.916
Fold 9 | Acc:0.821 | Kappa:0.706 | ROC:0.957
Fold 10 | Acc:0.846 | Kappa:0.744 | ROC:0.948

FINAL RESULT: word
Accuracy : 0.817258883248731
Kappa    : 0.7021138074786571

TASK: pword
pword total data: 350
Fold 1 | Acc:0.829 | Kappa:0.715 | ROC:0.925
Fold 2 | Acc:0.829 | Kappa:0.722 | ROC:0.967
Fold 3 | Acc:0.886 | Kappa:0.805 | ROC:0.986
Fold 4 | Acc:0.686 | Kappa:0.456 | ROC:0.880
Fold 5 | Acc:0.743 | Kappa:0.569 | ROC:0.903
Fold 6 | Acc:0.771 | Kappa:0.614 | ROC:0.903
Fold 7 | Acc:0.829 | Kappa:0.714 | ROC:0.953
Fold 8 | Acc:0.829 | Kappa:0.707 | ROC:0.919
Fold 9 | Acc:0.914 | Kappa:0.853 | ROC:0.9

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/DysgraphiaDB3/Observasi/EfficienNetB7'

In [ ]:
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/EfficientB7/EfficienNetB7_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")


===== FINAL RESULT (MEAN 10-FOLD) =====
       fold  accuracy  precision    recall        f1     kappa   roc_auc
task                                                                    
dword   5.5  0.790909   0.804295  0.790909  0.776655  0.620309  0.908664
pword   5.5  0.820000   0.837161  0.820000  0.814590  0.696140  0.930210
word    5.5  0.817308   0.832583  0.817308  0.817521  0.702349  0.934683

ALL DONE ✔


**EfficientNet V2 S**

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat

# =========================
# CNN FEATURE
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(paths, model):
    model.eval()
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())
    return np.array(feats)

def load_data():

    Xo, Xi, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)
            i_path = os.path.join(offline_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue

                base = os.path.splitext(f)[0]
                img = os.path.join(i_path, base + ".png")

                if not os.path.exists(img):
                    continue

                try:
                    feat = extract_online_features(os.path.join(o_path,f))

                    Xo.append(feat)
                    Xi.append(img)
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), Xi, np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online, X_img, y = load_data()

cnn = CNNFeatureExtractor().to(device)

#skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(X_img):
      #parts = p.split("/")
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]
    imgs = [X_img[i] for i in idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])

        svm_o = SVC(**svm_online_params[task], probability=True)
        svm_o.fit(Xtr,y_t[tr])

        po_tr = svm_o.predict_proba(Xtr)
        po_te = svm_o.predict_proba(Xte)

        # =========================
        # OFFLINE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr],cnn)
        Xe = extract_cnn([imgs[i] for i in te],cnn)

        sc2 = StandardScaler()
        Xt = sc2.fit_transform(Xt)
        Xe = sc2.transform(Xe)

        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt,y_t[tr])

        pc_tr = svm_c.predict_proba(Xt)
        pc_te = svm_c.predict_proba(Xe)

        # =========================
        # META
        # =========================
        meta_tr = np.hstack([po_tr,pc_tr])
        meta_te = np.hstack([po_te,pc_te])

        meta = SVC(kernel='rbf',C=1,probability=True)
        meta.fit(meta_tr,y_t[tr])

        pred = meta.predict(meta_te)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = meta.predict_proba(meta_te)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Efficients/Efficients_cm_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Efficients/Efficients_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Efficients/Efficiens_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 113MB/s]



TASK: word
word total data: 394
Fold 1 | Acc:0.825 | Kappa:0.705 | ROC:0.948
Fold 2 | Acc:0.750 | Kappa:0.598 | ROC:0.834
Fold 3 | Acc:0.775 | Kappa:0.627 | ROC:0.947
Fold 4 | Acc:0.800 | Kappa:0.681 | ROC:0.944
Fold 5 | Acc:0.795 | Kappa:0.677 | ROC:0.919
Fold 6 | Acc:0.846 | Kappa:0.741 | ROC:0.967
Fold 7 | Acc:0.795 | Kappa:0.660 | ROC:0.899
Fold 8 | Acc:0.769 | Kappa:0.627 | ROC:0.907
Fold 9 | Acc:0.795 | Kappa:0.661 | ROC:0.930
Fold 10 | Acc:0.769 | Kappa:0.610 | ROC:0.927

FINAL RESULT: word
Accuracy : 0.7918781725888325
Kappa    : 0.6583513985089622

TASK: pword
pword total data: 350
Fold 1 | Acc:0.686 | Kappa:0.478 | ROC:0.900
Fold 2 | Acc:0.800 | Kappa:0.672 | ROC:0.948
Fold 3 | Acc:0.829 | Kappa:0.699 | ROC:0.969
Fold 4 | Acc:0.886 | Kappa:0.808 | ROC:0.958
Fold 5 | Acc:0.886 | Kappa:0.811 | ROC:0.982
Fold 6 | Acc:0.771 | Kappa:0.608 | ROC:0.940
Fold 7 | Acc:0.829 | Kappa:0.729 | ROC:0.979
Fold 8 | Acc:0.857 | Kappa:0.765 | ROC:0.971
Fold 9 | Acc:0.743 | Kappa:0.566 | ROC:0.

**Resnet50**

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat

# =========================
# CNN FEATURE
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(pretrained=True)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(paths, model):
    model.eval()
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())
    return np.array(feats)

def load_data():

    Xo, Xi, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)
            i_path = os.path.join(offline_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue

                base = os.path.splitext(f)[0]
                img = os.path.join(i_path, base + ".png")

                if not os.path.exists(img):
                    continue

                try:
                    feat = extract_online_features(os.path.join(o_path,f))

                    Xo.append(feat)
                    Xi.append(img)
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), Xi, np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online, X_img, y = load_data()

cnn = CNNFeatureExtractor().to(device)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(X_img):
      #parts = p.split("/")
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]
    imgs = [X_img[i] for i in idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])

        svm_o = SVC(**svm_online_params[task], probability=True)
        svm_o.fit(Xtr,y_t[tr])

        po_tr = svm_o.predict_proba(Xtr)
        po_te = svm_o.predict_proba(Xte)

        # =========================
        # OFFLINE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr],cnn)
        Xe = extract_cnn([imgs[i] for i in te],cnn)

        sc2 = StandardScaler()
        Xt = sc2.fit_transform(Xt)
        Xe = sc2.transform(Xe)

        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt,y_t[tr])

        pc_tr = svm_c.predict_proba(Xt)
        pc_te = svm_c.predict_proba(Xe)

        # =========================
        # META
        # =========================
        meta_tr = np.hstack([po_tr,pc_tr])
        meta_te = np.hstack([po_te,pc_te])

        meta = SVC(kernel='rbf',C=1,probability=True)
        meta.fit(meta_tr,y_t[tr])

        pred = meta.predict(meta_te)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = meta.predict_proba(meta_te)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Resnet50/cm_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Resnet50/Resnet50_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Resnet50/cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



TASK: word
word total data: 394
Fold 1 | Acc:0.850 | Kappa:0.752 | ROC:0.979
Fold 2 | Acc:0.725 | Kappa:0.555 | ROC:0.842
Fold 3 | Acc:0.800 | Kappa:0.670 | ROC:0.943
Fold 4 | Acc:0.875 | Kappa:0.803 | ROC:0.974
Fold 5 | Acc:0.795 | Kappa:0.681 | ROC:0.919
Fold 6 | Acc:0.846 | Kappa:0.741 | ROC:0.970
Fold 7 | Acc:0.821 | Kappa:0.707 | ROC:0.930
Fold 8 | Acc:0.795 | Kappa:0.671 | ROC:0.913
Fold 9 | Acc:0.821 | Kappa:0.706 | ROC:0.953
Fold 10 | Acc:0.821 | Kappa:0.697 | ROC:0.935

FINAL RESULT: word
Accuracy : 0.8147208121827412
Kappa    : 0.6980557655160831

TASK: pword
pword total data: 350
Fold 1 | Acc:0.771 | Kappa:0.611 | ROC:0.922
Fold 2 | Acc:0.886 | Kappa:0.810 | ROC:0.981
Fold 3 | Acc:0.743 | Kappa:0.515 | ROC:0.977
Fold 4 | Acc:0.800 | Kappa:0.656 | ROC:0.949
Fold 5 | Acc:0.800 | Kappa:0.675 | ROC:0.963
Fold 6 | Acc:0.829 | Kappa:0.714 | ROC:0.931
Fold 7 | Acc:0.857 | Kappa:0.764 | ROC:0.979
Fold 8 | Acc:0.857 | Kappa:0.756 | ROC:0.973
Fold 9 | Acc:0.914 | Kappa:0.854 | ROC:0.

Resnet18

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA - META LEARNING (STACKING)
# ONLINE + OFFLINE + ROC-AUC + FULL METRICS
# STRATIFIED 10-FOLD CV
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"

# =========================
# SVM PARAMETER (FIXED)
# =========================
svm_online_params = {
    "pword": {"C":4, "gamma":0.015625, "kernel":"rbf"},
    "word": {"C":16, "gamma":0.015625, "kernel":"rbf"},
    "dword": {"C":4, "gamma":0.015625, "kernel":"rbf"}
}

svm_offline_params = {
    "pword": {"C":0.001953125, "gamma":0.015625, "kernel":"poly"},
    "word": {"C":0.001953125, "gamma":0.001953125, "kernel":"poly"},
    "dword": {"C":0.00390625, "gamma":0.0078125, "kernel":"poly"}
}

# =========================
# SAFE STATS
# =========================
# =========================================================
# SAFE STATISTICS (7 FEATURES)
# =========================================================
def safe_confusion_matrix(y_true, y_pred, labels=[0,1,2]):
  try:
    if len(np.unique(y_true)) == 0:
      return np.zeros((len(labels), len(labels)), dtype=int)
    return confusion_matrix(y_true, y_pred, labels=labels)
  except ValueError:
    return np.zeros((len(labels), len(labels)), dtype=int)

def stats(x):
    x = np.nan_to_num(np.array(x, dtype=np.float64),
                      nan=0.0, posinf=0.0, neginf=0.0)
    if len(x) < 2:
        return [0.0]*7
    return [
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        np.percentile(x, 5),
        np.percentile(x, 95)
    ]

# =========================================================
# SAFE DIFFERENCE
# =========================================================
def diff(a, t):
    a = np.array(a, dtype=np.float64)
    t = np.array(t, dtype=np.float64)
    if len(a) < 2:
        return np.zeros(1)
    dt = np.diff(t) + 1e-6
    return np.diff(a) / dt

# =========================================================
# MAIN FEATURE EXTRACTION
# =========================================================
def extract_online_features(path):

    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.replace([np.inf, -np.inf], 0).fillna(0)

    x = df.iloc[:,0].values
    y = df.iloc[:,1].values
    t = df.iloc[:,2].values + 1e-6
    status = df.iloc[:,3].values

    az = df.iloc[:,4].values
    alt = df.iloc[:,5].values
    pres = df.iloc[:,6].values

    # =========================================================
    # FEATURE FUNCTION PER STATUS
    # =========================================================
    def build(mask):

        xs = x[mask]
        ys = y[mask]
        ts = t[mask]

        if len(xs) < 3:
            return [0.0]*77  # fixed block size

        vx = diff(xs, ts)
        vy = diff(ys, ts)
        v  = np.sqrt(vx**2 + vy**2)

        ax = diff(vx, ts[:-1])
        ay = diff(vy, ts[:-1])
        a  = np.sqrt(ax**2 + ay**2)

        jx = diff(ax, ts[:-2])
        jy = diff(ay, ts[:-2])

        speed = np.sqrt(xs**2 + ys**2)

        feats = []

        # =========================
        # POSITION (14)
        # =========================
        feats += stats(xs)
        feats += stats(ys)

        # =========================
        # VELOCITY (21)
        # =========================
        feats += stats(vx)
        feats += stats(vy)
        feats += stats(v)

        # =========================
        # ACCELERATION (21)
        # =========================
        feats += stats(ax)
        feats += stats(ay)
        feats += stats(a)

        # =========================
        # JERK (14)
        # =========================
        feats += stats(jx)
        feats += stats(jy)

        # =========================
        # GEOMETRIC MOTION (7)
        # =========================
        feats += stats(speed)

        return feats

    # =========================================================
    # SURFACE (pen down = 1)
    # =========================================================
    surface = build(status == 1)

    # =========================================================
    # AIR (pen up = 0)
    # =========================================================
    air = build(status == 0)

    # =========================================================
    # SENSOR (ONLY PEN DOWN)
    # =========================================================
    mask = status == 1

    sensor = []
    sensor += stats(alt[mask])
    sensor += stats(az[mask])
    sensor += stats(pres[mask])

    # =========================================================
    # GLOBAL FEATURES
    # =========================================================
    width = np.max(x) - np.min(x)
    height = np.max(y) - np.min(y)
    duration = t[-1] - t[0]
    total_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
    pen_lift = np.sum((status[:-1] == 1) & (status[1:] == 0))

    global_feat = [
        width,
        height,
        duration,
        total_length,
        pen_lift
    ]

    # =========================================================
    # FINAL FEATURE VECTOR
    # =========================================================
    feat = surface + air + sensor + global_feat

    feat = np.nan_to_num(np.array(feat, dtype=np.float64),
                         nan=0.0, posinf=0.0, neginf=0.0)

    return feat

# =========================
# CNN FEATURE
# =========================
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(paths, model):
    model.eval()
    feats = []
    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())
    return np.array(feats)

def load_data():

    Xo, Xi, y = [], [], []

    for task in ["word","pword","dword"]:

        online_path = os.path.join(ONLINE_ROOT, task)
        offline_path = os.path.join(OFFLINE_ROOT, task)

        for cls in os.listdir(online_path):

            o_path = os.path.join(online_path, cls)
            i_path = os.path.join(offline_path, cls)

            for f in os.listdir(o_path):

                if not f.endswith(".txt"):
                    continue

                base = os.path.splitext(f)[0]
                img = os.path.join(i_path, base + ".png")

                if not os.path.exists(img):
                    continue

                try:
                    feat = extract_online_features(os.path.join(o_path,f))

                    Xo.append(feat)
                    Xi.append(img)
                    label_map = {"normal":0,"grade1":1,"grade2":2}
                    y.append(label_map[cls])


                except:
                    continue

    return np.array(Xo), Xi, np.array(y)

# =========================
# MAIN PIPELINE
# =========================
X_online, X_img, y = load_data()

cnn = CNNFeatureExtractor().to(device)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
cms = []

# =========================
# LOOP TASK
# =========================
for task in ["word","pword","dword"]:

    print("\n====================")
    print("TASK:",task)
    print("====================")

    # FILTER (perkelas data latih)
    idx = []
    for i, p in enumerate(X_img):
      #parts = p.split("/")
      parts = p.replace("\\", "/").split("/")
      if task in parts:
        idx.append(i)

    # FILTER (data latih : semua task)
    #idx = [i for i,p in enumerate(X_img) if f"/{task}/" in p]
    print(f"{task} total data:", len(idx))

    if len(idx) == 0:
        print("WARNING: data kosong, skip")
        continue

    X = X_online[idx]
    y_t = y[idx]
    imgs = [X_img[i] for i in idx]

    y_true_all = []
    y_pred_all = []

    for fold,(tr,te) in enumerate(skf.split(X,y_t),1):

        if len(te) == 0:
            print(f"Fold {fold} kosong, skip")
            continue

        # =========================
        # ONLINE
        # =========================
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])

        svm_o = SVC(**svm_online_params[task], probability=True)
        svm_o.fit(Xtr,y_t[tr])

        po_tr = svm_o.predict_proba(Xtr)
        po_te = svm_o.predict_proba(Xte)

        # =========================
        # OFFLINE
        # =========================
        Xt = extract_cnn([imgs[i] for i in tr],cnn)
        Xe = extract_cnn([imgs[i] for i in te],cnn)

        sc2 = StandardScaler()
        Xt = sc2.fit_transform(Xt)
        Xe = sc2.transform(Xe)

        svm_c = SVC(**svm_offline_params[task], probability=True)
        svm_c.fit(Xt,y_t[tr])

        pc_tr = svm_c.predict_proba(Xt)
        pc_te = svm_c.predict_proba(Xe)

        # =========================
        # META
        # =========================
        meta_tr = np.hstack([po_tr,pc_tr])
        meta_te = np.hstack([po_te,pc_te])

        meta = SVC(kernel='rbf',C=1,probability=True)
        meta.fit(meta_tr,y_t[tr])

        pred = meta.predict(meta_te)

        y_true = y_t[te]

        # =========================
        # METRICS
        # =========================
        acc = accuracy_score(y_true,pred)
        prec = precision_score(y_true,pred,average='weighted',zero_division=0)
        rec = recall_score(y_true,pred,average='weighted',zero_division=0)
        f1 = f1_score(y_true,pred,average='weighted',zero_division=0)
        kappa = cohen_kappa_score(y_true,pred)

        # =========================
        # CONFUSION MATRIX FIX
        # =========================
        #cm = confusion_matrix(y_true, pred, labels=[0,1,2])
        #cm = confusion_matrix(y_true, pred)
        cm = safe_confusion_matrix(y_true, pred, labels=[0,1,2])

        # =========================
        # ROC FIX
        # =========================
        proba = meta.predict_proba(meta_te)

        if len(np.unique(y_true)) < 2:
            roc = np.nan
        else:
            try:
                roc = roc_auc_score(
                    y_true,
                    proba,
                    multi_class='ovr',
                    average='weighted'
                )
            except:
                roc = np.nan

        print(f"Fold {fold} | Acc:{acc:.3f} | Kappa:{kappa:.3f} | ROC:{roc:.3f}")

        results.append({
            "task": task,
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "kappa": kappa,
            "roc_auc": roc
        })

        cms.append({
            "task": task,
            "fold": fold,
            "cm": cm
        })

        # SAVE CM PER FOLD
        pd.DataFrame(cm,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Resnet18/Res18_cm_{task}_fold{fold}.csv")

        y_true_all.extend(y_true)
        y_pred_all.extend(pred)

    # =========================
    # FINAL PER TASK
    # =========================
    print("\nFINAL RESULT:",task)
    print("Accuracy :",accuracy_score(y_true_all,y_pred_all))
    print("Kappa    :",cohen_kappa_score(y_true_all,y_pred_all))

# =========================
# SAVE METRICS
# =========================
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/DysgraphiaDB3/Observasi/Resnet18/Res18_meta_metrics.csv", index=False)

# =========================
# SAVE CM TOTAL
# =========================
for t in ["word","pword","dword"]:

    cms_task = [c["cm"] for c in cms if c["task"] == t]

    if len(cms_task) > 0:
        cm_sum = np.sum(cms_task, axis=0)

        pd.DataFrame(cm_sum,
            index=[0,1,2],
            columns=[0,1,2]
        ).to_csv(f"/content/drive/MyDrive/DysgraphiaDB3/Observasi/Resnet18/Res18_cm_{t}_total.csv")

# =========================
# FINAL RESULT
# =========================
print("\n===== FINAL RESULT (MEAN 10-FOLD) =====")
print(df.groupby("task").mean(numeric_only=True))

print("\nALL DONE ✔")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 115MB/s]



TASK: word
word total data: 394
Fold 1 | Acc:0.825 | Kappa:0.705 | ROC:0.960
Fold 2 | Acc:0.725 | Kappa:0.555 | ROC:0.827
Fold 3 | Acc:0.775 | Kappa:0.626 | ROC:0.931
Fold 4 | Acc:0.825 | Kappa:0.719 | ROC:0.938
Fold 5 | Acc:0.795 | Kappa:0.681 | ROC:0.911
Fold 6 | Acc:0.846 | Kappa:0.739 | ROC:0.971
Fold 7 | Acc:0.821 | Kappa:0.707 | ROC:0.906
Fold 8 | Acc:0.718 | Kappa:0.534 | ROC:0.907
Fold 9 | Acc:0.795 | Kappa:0.661 | ROC:0.928
Fold 10 | Acc:0.795 | Kappa:0.650 | ROC:0.949

FINAL RESULT: word
Accuracy : 0.7918781725888325
Kappa    : 0.6575002650270327

TASK: pword
pword total data: 350
Fold 1 | Acc:0.800 | Kappa:0.668 | ROC:0.954
Fold 2 | Acc:0.914 | Kappa:0.858 | ROC:0.967
Fold 3 | Acc:0.857 | Kappa:0.760 | ROC:0.974
Fold 4 | Acc:0.743 | Kappa:0.584 | ROC:0.937
Fold 5 | Acc:0.714 | Kappa:0.559 | ROC:0.924
Fold 6 | Acc:0.771 | Kappa:0.619 | ROC:0.921
Fold 7 | Acc:0.857 | Kappa:0.768 | ROC:0.950
Fold 8 | Acc:0.800 | Kappa:0.680 | ROC:0.969
Fold 9 | Acc:0.857 | Kappa:0.764 | ROC:0.

bi LSTM + DenceNet121  

In [ ]:
# =========================================================
# MULTIMODAL DYSGRAPHIA FULL PIPELINE
# ONLINE + OFFLINE + CNN + BiLSTM + FUSION + SAVE
# =========================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    roc_auc_score
)

# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ONLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Online"
OFFLINE_ROOT = "/content/drive/MyDrive/DysgraphiaDB3/Offline_images"
OUT_DIR = "/content/drive/MyDrive/DysgraphiaDB3/Results"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# LABEL ENCODER
# =========================================================
le = LabelEncoder()

# =========================================================
# ONLINE LOADER (FIXED 9 FEATURES)
# =========================================================
def load_online():

    X, y = [], []

    for task in ["word","pword","dword"]:
        task_path = os.path.join(ONLINE_ROOT, task)

        for cls in os.listdir(task_path):
            cls_path = os.path.join(task_path, cls)

            for f in os.listdir(cls_path):
                if not f.endswith(".txt"):
                    continue

                df = pd.read_csv(os.path.join(cls_path,f),
                                 sep=r"\s+", header=None).fillna(0)

                x = df.iloc[:,0].values
                y_ = df.iloc[:,1].values
                t = df.iloc[:,2].values + 1e-6

                pen = df.iloc[:,3].values if df.shape[1]>3 else np.zeros(len(x))
                az  = df.iloc[:,4].values if df.shape[1]>4 else np.zeros(len(x))
                al  = df.iloc[:,5].values if df.shape[1]>5 else np.zeros(len(x))
                pr  = df.iloc[:,6].values if df.shape[1]>6 else np.zeros(len(x))

                dx = np.diff(x, prepend=x[0])
                dy = np.diff(y_, prepend=y_[0])

                vx = dx / (np.diff(t, prepend=t[0]) + 1e-6)
                vy = dy / (np.diff(t, prepend=t[0]) + 1e-6)
                v  = np.sqrt(vx**2 + vy**2)

                seq = np.stack([
                    x, y_,
                    vx, vy, v,
                    pen, pr, al, az
                ], axis=1)

                X.append(seq)
                y.append(cls)

    return X, np.array(y)

# =========================================================
# PADDING
# =========================================================
def pad(seqs, max_len=500):
    F = seqs[0].shape[1]
    out = []

    for s in seqs:
        if len(s) > max_len:
            s = s[:max_len]
        else:
            s = np.vstack([s, np.zeros((max_len-len(s),F))])
        out.append(s)

    return np.array(out)

# =========================================================
# BI-LSTM (ONLINE)
# =========================================================
class BiLSTM(nn.Module):
    def __init__(self, input_dim=9):
        super().__init__()

        self.lstm = nn.LSTM(input_dim,128,
                            batch_first=True,
                            bidirectional=True)

        self.fc = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

    def forward(self,x):
        out,_ = self.lstm(x)
        out = out[:,-1,:]
        return self.fc(out)

# =========================================================
# CNN (OFFLINE IMAGE)
# =========================================================
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self,x):
        x = self.features(x)
        x = self.pool(x)
        return torch.flatten(x,1)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_cnn(model, paths):
    model.eval()
    feats = []

    with torch.no_grad():
        for p in paths:
            img = Image.open(p).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)
            feats.append(model(img).cpu().numpy().flatten())

    return np.array(feats)

# =========================================================
# OFFLINE LOADER
# =========================================================
def load_offline():

    paths, labels = [], []

    for task in ["word","pword","dword"]:
        for cls in os.listdir(os.path.join(OFFLINE_ROOT,task)):

            path = os.path.join(OFFLINE_ROOT,task,cls)

            for f in os.listdir(path):
                if f.endswith(".png"):
                    paths.append(os.path.join(path,f))
                    labels.append(cls)

    return paths, np.array(labels)

# =========================================================
# LOAD DATA
# =========================================================
X_seq, y_seq = load_online()
X_seq = pad(X_seq)

img_paths, y_img = load_offline()

y = le.fit_transform(y_seq)

# =========================================================
# SPLIT
# =========================================================
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=y, random_state=42
)

# =========================================================
# ONLINE FEATURE
# =========================================================
bilstm = BiLSTM().to(device)

X_train_seq = torch.tensor(X_seq[train_idx],dtype=torch.float32).to(device)
X_test_seq  = torch.tensor(X_seq[test_idx],dtype=torch.float32).to(device)

with torch.no_grad():
    f_train_online = bilstm(X_train_seq).cpu().numpy()
    f_test_online  = bilstm(X_test_seq).cpu().numpy()

# =========================================================
# OFFLINE FEATURE
# =========================================================
cnn = CNN().to(device)

img_train = [img_paths[i] for i in train_idx if i < len(img_paths)]
img_test  = [img_paths[i] for i in test_idx if i < len(img_paths)]

f_train_img = extract_cnn(cnn, img_train)
f_test_img  = extract_cnn(cnn, img_test)

# =========================================================
# ALIGN
# =========================================================
n_train = min(len(f_train_online), len(f_train_img))
n_test  = min(len(f_test_online), len(f_test_img))

X_train = np.hstack([
    f_train_online[:n_train],
    f_train_img[:n_train]
])

X_test = np.hstack([
    f_test_online[:n_test],
    f_test_img[:n_test]
])

y_train = y[train_idx][:n_train]
y_test  = y[test_idx][:n_test]

# =========================================================
# FUSION MODEL
# =========================================================
clf = SVC(kernel='rbf', probability=True)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)

# =========================================================
# METRICS
# =========================================================
print("Accuracy:", accuracy_score(y_test,pred))
print("Kappa:", cohen_kappa_score(y_test,pred))
print("\nReport:\n", classification_report(y_test,pred))
print("\nCM:\n", confusion_matrix(y_test,pred))
print("\nROC-AUC:", roc_auc_score(y_test, prob, multi_class='ovr'))

# =========================================================
# SAVE ALL OUTPUT
# =========================================================
pd.DataFrame({
    "y_true": y_test,
    "y_pred": pred
}).to_csv(os.path.join(OUT_DIR,"predictions.csv"), index=False)

np.savetxt(os.path.join(OUT_DIR,"confusion_matrix.csv"),
           confusion_matrix(y_test,pred),
           delimiter=",")

pd.DataFrame(prob).to_csv(os.path.join(OUT_DIR,"probabilities.csv"), index=False)

pd.DataFrame([{
    "accuracy": accuracy_score(y_test,pred),
    "kappa": cohen_kappa_score(y_test,pred),
    "roc_auc": roc_auc_score(y_test, prob, multi_class='ovr')
}]).to_csv(os.path.join(OUT_DIR,"metrics.csv"), index=False)

np.save(os.path.join(OUT_DIR,"online_features.npy"), f_test_online)
np.save(os.path.join(OUT_DIR,"offline_features.npy"), f_test_img)
np.save(os.path.join(OUT_DIR,"fused_features.npy"), X_test)

print("\n✔ SEMUA HASIL SUDAH DISIMPAN DI:", OUT_DIR)

**Heatmap Confusion Matrix**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

BASE_PATH = "/content/drive/MyDrive/DysgraphiaDB3/Observasi/EfficientB7/"
tasks = ["word", "pword", "dword"]

labels = ["normal", "grade1", "grade2"]

def plot_cm(cm, title, normalize=False):

    plt.figure(figsize=(6,5))

    if normalize:
        cm = cm.div(cm.sum(axis=1), axis=0)

        sns.heatmap(
            cm,
            annot=True,
            fmt=".2f",
            cmap="Blues",
            linewidths=0.5,
            linecolor="black"
        )
    else:
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            linewidths=0.5,
            linecolor="black"
        )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)

    plt.tight_layout()
    plt.show()

    for task in tasks:

    cm_path = os.path.join(BASE_PATH, f"cm_{task}_total.csv")

    if not os.path.exists(cm_path):
        print(f"❌ File not found: {cm_path}")
        continue

    # LOAD CSV
    cm = pd.read_csv(cm_path, index_col=0)

    # LABELING BIAR PAPER-READY
    cm.index = labels
    cm.columns = labels

    print("\n==============================")
    print("TASK:", task)
    print("==============================")
    print("\nRAW CONFUSION MATRIX")
    print(cm)

    # =========================
    # RAW HEATMAP
    # =========================
    plot_cm(
        cm,
        title=f"{task.upper()} - Confusion Matrix (Raw Counts)",
        normalize=False
    )

    # =========================
    # NORMALIZED HEATMAP
    # =========================
    plot_cm(
        cm,
        title=f"{task.upper()} - Confusion Matrix (Normalized)",
        normalize=True
    )

IndentationError: expected an indented block after 'for' statement on line 43 (879567640.py, line 45)